# 05. Super Resolution - ESPCN_X3

| 항목 | 내용 |
|---|---|
| 대상 모델 | `ESPCN_X3.dxnn` |
| 카테고리 | **Super Resolution** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 9채널 출력이 1채널 3배 영상으로 재배열되는 규칙을 그림 없이 좌표 계산으로 설명할 수 있다.
2. 밝기 채널만 모델에 넣고 색은 확대해 합치는 이유를 사람 눈의 특성과 연결해 설명할 수 있다.
3. 확대 배율을 출력 크기에서 자동으로 알아내는 계산을 직접 수행할 수 있다.
4. 단순 확대와 모델 확대의 차이를 선명도 수치로 비교해 제시할 수 있다.

## 1. Super Resolution (초해상도)

**작은 영상을 받아 픽셀 수를 늘려 큰 영상을 만드는 작업**

단순 확대는 이미 있는 값 사이를 이어 붙여 빈 칸을 메우므로 윤곽이 뭉개진다. 이 작업은 학습한 무늬를 근거로
없던 값을 만들어 넣어 윤곽을 살린다.

**출력의 성격**

픽셀마다 값이 하나씩 나오는 영상이며 크기가 입력의 정해진 배수다. 없던 정보를 만들어 채우므로 결과가
그럴듯해 보이더라도 원본에 있던 것과 같다고 보장할 수 없다. 판독·감정처럼 사실이 중요한 자리에서는 이
성질을 반드시 밝혀야 한다.

**활용 사례**

- 저해상도 영상의 화면 표시 품질 개선
- 오래된 사진·영상 자료의 확대
- 작게 잘린 관심 영역을 키워 사람이 확인

**포함되지 않는 작업**

- 잡티 제거 (Denoising)
- 어두운 영상 밝히기 (Low-light Enhancement)
- 흔들린 영상 되살리기 (Deblurring)

## 2. 전체 파이프라인

```text
sample_superresolution.png  768×512 BGR
  → 전처리         BGR→GRAY · 17×17 로 축소 · 채널 축 추가 · uint8 · 원본 BGR 보관
  → run()          입력 Tensor 하나를 NPU 에 전달
  → npu_0          Conv_output_0  [1, 9, 17, 17]  float32  (내부 전달, Public 아님)
  → cpu_0          output  [1, 1, 51, 51]  float32  (Public Output — 9채널을 3×3 으로 재배열)
  → 후처리         squeeze → 배율 판정 → clip(0,1) → ×255 → 원본 색과 합침
  → Result         SuperResolutionResult (51×51 컬러 영상 · scale_factor 3)
  → 시각화         단순 확대본과 좌우로 붙여 비교
```

![05. Super Resolution 전체 파이프라인](resources/05_Super_Resolution_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input` |
| Shape / Dtype | `[1, 17, 17, 1]` / `uint8` |
| 채널 순서 | NHWC, 채널 1개 |
| Color / 값 범위 | Grayscale (밝기 채널) / 0~255 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | 없음 |

**입력이 17×17 로 고정돼 있다는 점이 이 모델을 쓰는 방식을 정한다.**

- **원본을 17×17 로 줄여서 넣는다** — 768×512 사진이 289개 픽셀로 줄어든 뒤 3배로 커진다. 결과 51×51 은
  원본보다 훨씬 작다
- **큰 사진을 키우려면 타일로 쪼개 반복해야 한다** — 이 예제에는 그 단계가 없다. 한 번의 실행은 17×17
  조각 하나를 다루는 것과 같다
- **밝기 한 채널만 받는다** — 색은 모델을 거치지 않는다

289 바이트짜리 입력은 이 저장소의 17개 중 가장 작다. 모델이 다루는 단위가 사진 한 장이 아니라 작은 조각
하나라는 뜻이다.

### 3.2 출력

이 모델은 `.dxnn` 하나 안에 Task 가 둘이며 **재배열이 CPU Task 의 일**이다.

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `Conv_output_0` | `[1, 9, 17, 17]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `cpu_0` | `output` | `[1, 1, 51, 51]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. 9채널 값은 밖으로 나오지 않으므로
노트북에서 볼 수 없고, 재배열이 끝난 `output` 만 손에 들어온다. 두 tensor 의 바이트 크기가 같은 것은 값의
개수가 같기 때문이다 — `9 × 17 × 17 = 2601` 이고 `1 × 51 × 51 = 2601` 이다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 1 | 채널. 밝기 한 장 |
| 2 | 51 | 높이. `17 × 3` |
| 3 | 51 | 너비. `17 × 3` |

**9채널이 3배 영상이 되는 규칙**

확대를 하려면 픽셀을 더 만들어야 한다. ESPCN 은 크기를 늘리는 대신 **채널을 배율의 제곱만큼 늘려 놓고
마지막에 그 채널을 픽셀 자리로 흩는다.** 3배 확대이므로 채널이 `3 × 3 = 9` 개다.

```text
채널 9장, 각 17×17            결과 51×51
  ch0 ch1 ch2                 한 칸(17×17 중 하나)이 3×3 블록이 된다
  ch3 ch4 ch5      →          왼쪽 위부터 ch0 ch1 ch2
  ch6 ch7 ch8                              ch3 ch4 ch5
                                           ch6 ch7 ch8
```

좌표로 쓰면 결과 픽셀 `(y, x)` 의 값은 채널 `(y mod 3) × 3 + (x mod 3)` 의 `(y // 3, x // 3)` 자리에서 온다.
예를 들어 결과 `(4, 5)` 는 `4 mod 3 = 1`, `5 mod 3 = 2` 이므로 채널 `1 × 3 + 2 = 5` 의 `(1, 1)` 값이다.
이 재배열을 sub-pixel convolution 또는 pixel shuffle 이라 부르며, **값을 새로 계산하지 않고 자리만 옮기므로
곱셈이 들어가지 않는다.** 확대를 마지막에 두어 앞의 모든 계산을 작은 크기에서 하는 것이 이 구조의 이득이다.

#### 출력을 사람이 읽는 값으로 바꾸기

출력은 0~1 부근의 밝기 한 장이고 화면에 그릴 영상은 0~255 컬러다. 되돌리는 순서는 아래와 같다.

| 순서 | 하는 일 | 이유 |
|---|---|---|
| 1 | 배율 판정 | 출력 높이를 입력 높이로 나눠 정수로 반올림한다. `51 / 17 = 3` |
| 2 | `clip(0, 1)` | 범위를 벗어난 값이 되돌릴 때 넘치지 않게 한다 |
| 3 | `×255` 후 `uint8` | 밝기 채널을 정수로 바꾼다 |
| 4 | 색 합치기 | 원본의 색 성분을 결과 크기로 확대해 밝기와 합친다 |

**모델이 만든 것은 밝기뿐이므로 색은 원본에서 가져온다.** 원본을 밝기(Y)와 색(Cr·Cb) 성분으로 나누고,
색 두 장만 51×51 로 확대해 모델이 만든 밝기와 합친 뒤 BGR 로 되돌린다.

이렇게 해도 되는 이유는 사람 눈이 **밝기의 변화에는 민감하고 색의 변화에는 둔감**하기 때문이다. 색을 단순
확대해 뭉개도 윤곽은 밝기가 만들므로 결과가 선명하게 보인다. 영상 압축이 색 성분을 밝기보다 적게 저장하는
것도 같은 성질을 쓴 것이다.

이번 실행의 값은 최소 `0.0000`, 최대 `1.0000` 이라 `clip` 이 잘라 내는 픽셀이 없다. 평균 `0.4100` 은
`0.4100 × 255 = 104.6` 이므로 `uint8` 104 가 된다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. `npu_0` 이 9채널
17×17 을 내고 `cpu_0` 이 1채널 51×51 을 내는지, 둘의 바이트 크기가 같은지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "ESPCN_X3.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정한다. 이후 셀은 여기서 만든 이름을 그대로 쓴다.

In [ ]:
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_PATH = DX_APP / "sample" / "img/sample_superresolution.png"

print("input image:", IMAGE_PATH.name)

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 이 모델은 **원본 BGR 자체**를
  담아 두었다가 후처리에서 색을 가져다 쓴다
- `SuperResolutionResult` — 후처리가 만들어 시각화로 넘기는 결과. 확대된 영상과 배율을 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0
    original_image: np.ndarray = field(default=None)


@dataclass
class SuperResolutionResult:
    """확대된 영상과 배율. 원본 대응: common/base/i_processor.py"""

    output_image: np.ndarray = field(default=None)
    scale_factor: int = 2


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 입력이 17×17 이고 출력이 51×51 인 것을
확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

BGR 원본을 밝기 한 채널로 바꾸고 17×17 로 줄인 뒤 채널 축을 붙인다. **원본 BGR 을 그대로 Context 에 담아
두는 것이 이 모델의 특징**이며, 후처리가 거기서 색 성분을 가져다 쓴다.

확인할 것:

- 채널 — 3채널 BGR → 1채널 밝기
- 크기 — 768×512 원본이 17×17 로 줄어듦
- dtype — `uint8` 유지
- Context — 원본 BGR 보관 여부

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 이 사진 전체가 17×17 로 줄어든다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}")

#### 전처리 함수 정의

- `espcn_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `espcn_preprocessor_process` — 밝기 변환·축소·채널 축 추가를 거쳐 입력 Tensor 와 Context 로 변환

In [ ]:
def espcn_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def espcn_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)
    # 후처리가 색 성분을 가져다 쓰므로 원본 BGR 을 그대로 담아 둔다.
    ctx.original_image = input_image

    gray = cv2.cvtColor(input_image, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (ctx.input_width, ctx.input_height))

    return resized[:, :, np.newaxis], ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. 17×17 로 줄어든 그림을 크게 늘려 보면 확대의 출발점이 얼마나 거친지
드러난다.

In [ ]:
preprocessor_state = espcn_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = espcn_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"min={input_tensor.min()} max={input_tensor.max()}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"original_image 보관={ctx.original_image is not None}")
print(f"픽셀 수       {ctx.original_width * ctx.original_height:,} → "
      f"{INPUT_WIDTH * INPUT_HEIGHT:,}")

notebook_show_bgr(cv2.resize(input_tensor[:, :, 0], None, fx=12, fy=12,
                             interpolation=cv2.INTER_NEAREST),
                  f"Preprocessed {INPUT_WIDTH}x{INPUT_HEIGHT} - zoom x12", figsize=(6, 6))

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 NPU 에 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복
비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- shape — `[1, 1, 51, 51]` 로 이미 3배 커져 있다
- 값 개수 — 입력 289개가 2,601개로 늘었다
- 값 범위 — 0~1 부근이라 그대로 화면에 그릴 수 없다

In [ ]:
raw = np.asarray(outputs[0])
notebook_describe_array("raw output", raw)

flat = raw.reshape(-1)
print(f"값 개수  입력 {input_tensor.size:,} → 출력 {flat.size:,} ({flat.size // input_tensor.size}배)")
print(f"0 미만 = {int((flat < 0.0).sum())}개, 1 초과 = {int((flat > 1.0).sum())}개 / {flat.size}")

### 4.4 후처리

크기 1인 축을 눌러 `[51, 51]` 로 만들고 배율을 판정한 뒤 밝기를 정수로 바꾸고 원본 색을 합친다. 순서는
squeeze → 배율 판정 → `clip` → `×255` → 색 합치기이며 이 순서를 바꾸지 않는다.

**배율을 코드가 판정하는 것은 모델 파일이 배율을 알려 주지 않기 때문이다.** 출력 높이를 입력 높이로 나눠
정수로 반올림하며, 이번 실행은 `51 / 17 = 3` 이다. 설정으로 배율을 못 박을 수도 있으나 이 모델은 설정
파일이 비어 있어 자동 판정을 쓴다.

색을 합치는 단계에서 원본을 밝기와 색으로 나눈다. `cv2` 는 YCrCb 순서를 쓰므로 채널 1이 Cr, 채널 2가 Cb
이며, 이 두 장만 51×51 로 확대해 모델이 만든 밝기와 쌓은 뒤 BGR 로 되돌린다.

#### 후처리 함수 정의

- `espcn_postprocessor_create` — 배율을 state 에 담음. 0 이면 출력 크기에서 자동 판정
- `espcn_merge_color` — 모델이 만든 밝기에 원본 색을 확대해 합침
- `espcn_postprocessor_process` — squeeze·배율 판정·변환을 거쳐 Result 생성

In [ ]:
def espcn_postprocessor_create(input_height: int, scale_factor: int = 0) -> dict:
    """고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"input_height": input_height, "scale_factor": scale_factor}


def espcn_merge_color(sr_y: np.ndarray, original_bgr: np.ndarray) -> np.ndarray:
    """모델이 만든 밝기에 원본 색을 확대해 합친다. 원본 대응: ESPCNPostprocessor.process() 의 색 복원"""
    out_h, out_w = sr_y.shape
    ycrcb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2YCrCb)
    cr = cv2.resize(ycrcb[:, :, 1], (out_w, out_h), interpolation=cv2.INTER_CUBIC)
    cb = cv2.resize(ycrcb[:, :, 2], (out_w, out_h), interpolation=cv2.INTER_CUBIC)
    return cv2.cvtColor(np.stack([sr_y, cr, cb], axis=2), cv2.COLOR_YCrCb2BGR)


def espcn_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    output = np.squeeze(outputs[0])
    if output.ndim == 3:
        output = np.transpose(output, (1, 2, 0))

    scale = state["scale_factor"]
    if scale == 0:
        scale = max(1, round(output.shape[0] / state["input_height"]))

    output = np.clip(output, 0.0, 1.0)
    sr_y = (output * 255.0).astype(np.uint8)

    if output.ndim == 2 and ctx.original_image is not None:
        output_image = espcn_merge_color(sr_y, ctx.original_image)
    else:
        output_image = sr_y

    return [SuperResolutionResult(output_image=output_image, scale_factor=scale)]

#### 후처리 실행

확인할 것:

- `scale_factor` — 자동 판정 결과가 3 인지
- shape·dtype — `uint8` `[51, 51, 3]` 로 컬러가 되었는지
- 선명도 — 같은 배율로 단순 확대한 것과 비교해 얼마나 다른지

In [ ]:
postprocessor_state = espcn_postprocessor_create(INPUT_HEIGHT)
results = espcn_postprocessor_process(postprocessor_state, outputs, ctx)
sr_image = results[0].output_image

print(f"scale_factor  {results[0].scale_factor} (출력 {np.squeeze(raw).shape[0]} / 입력 {INPUT_HEIGHT})")
print(f"sr_image      shape={sr_image.shape} dtype={sr_image.dtype} mean={sr_image.mean():.2f}")

# 같은 배율로 단순 확대한 밝기와 비교한다. 선명도는 라플라시안 분산으로 잰다.
bicubic_y = cv2.resize(input_tensor[:, :, 0], (sr_image.shape[1], sr_image.shape[0]),
                       interpolation=cv2.INTER_CUBIC)
model_y = cv2.cvtColor(sr_image, cv2.COLOR_BGR2YCrCb)[:, :, 0]
print(f"|단순확대 - 모델| mean={np.abs(bicubic_y.astype(np.int16) - model_y.astype(np.int16)).mean():.2f}")
print(f"선명도(라플라시안 분산)  단순확대={cv2.Laplacian(bicubic_y, cv2.CV_64F).var():.1f} "
      f"모델={cv2.Laplacian(model_y, cv2.CV_64F).var():.1f}")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 원본을 결과 크기로 단순 확대한 것과 좌우로 붙인다. 두 그림의 크기를 같게 맞추므로 같은
배율에서 무엇이 다른지 바로 비교된다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본 폰트에 한글 자모가
없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def espcn_visualizer_visualize(image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 단순 확대본과 좌우로 붙여 그린다. 원본 대응: Runner.visualize()"""
    sr_image = results[0].output_image
    if sr_image.ndim == 2:
        sr_image = cv2.cvtColor(sr_image, cv2.COLOR_GRAY2BGR)

    sr_h, sr_w = sr_image.shape[:2]
    input_upscaled = cv2.resize(image, (sr_w, sr_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.hstack([input_upscaled, sr_image])

    cv2.putText(canvas, "Input (upscaled)", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (0, 255, 0), 2)
    cv2.putText(canvas, "Super-Resolved", (sr_w + 10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (0, 255, 0), 2)
    return canvas

#### 결과 시각화

좌우 비교를 inline 으로 표시한다. 51×51 은 화면에서 아주 작으므로 그림을 크게 늘려 표시한다. 왼쪽은 원본을
51×51 로 줄인 것이고 오른쪽이 모델 결과다. 왼쪽이 모델의 입력보다 좋은 출발점이라는 점은 다음 셀에서
바로잡는다.

In [ ]:
output_image = espcn_visualizer_visualize(image, results)
notebook_show_bgr(cv2.resize(output_image, None, fx=6, fy=6, interpolation=cv2.INTER_NEAREST),
                  "Simple downscale vs Super-Resolved (zoom x6)", figsize=(12, 6))

#### 같은 출발점에서 다시 비교

위 그림의 왼쪽은 **원본 768×512 를 51×51 로 줄인 것**이라 모델보다 유리한 출발점이다. 줄이는 과정에서
여러 픽셀이 평균되어 정보가 남기 때문이다. 모델은 17×17 만 보았으므로, 공정한 비교 대상은 **같은 17×17 을
단순 확대한 것**이다.

In [ ]:
fair_plain = cv2.cvtColor(
    cv2.resize(input_tensor[:, :, 0], (sr_image.shape[1], sr_image.shape[0]),
               interpolation=cv2.INTER_CUBIC), cv2.COLOR_GRAY2BGR)
fair_model = cv2.cvtColor(cv2.cvtColor(sr_image, cv2.COLOR_BGR2YCrCb)[:, :, 0],
                          cv2.COLOR_GRAY2BGR)
fair = cv2.resize(np.hstack([fair_plain, fair_model]), None, fx=6, fy=6,
                  interpolation=cv2.INTER_NEAREST)
cv2.putText(fair, "17x17 bicubic x3", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
cv2.putText(fair, "17x17 model x3", (fair.shape[1] // 2 + 10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
            (0, 255, 0), 2)

notebook_show_bgr(fair, "Same 17x17 input - bicubic vs model (zoom x6)", figsize=(12, 6))